# Detailed adaptive-rank attention evaluation
GPU-only, T4 guard. Private execution. Code pinned to original commit plus embedded follow-up sources. No credentials or private account data.


In [ ]:
import torch, subprocess, os, base64, pathlib
assert torch.cuda.is_available(), "CUDA unavailable; refusing CPU benchmark"
assert "T4" in torch.cuda.get_device_name(0), "Non-T4 GPU; refusing run"
print("HARDWARE", torch.cuda.get_device_name(0), torch.__version__)


In [ ]:
subprocess.run(["git", "clone", "https://github.com/Jaiadithya71/linear-attention-benchmark.git", "/kaggle/working/linear-attention-benchmark"], check=True)
os.chdir("/kaggle/working/linear-attention-benchmark")
subprocess.run(["git", "checkout", "68960ab"],check=True)
pathlib.Path('lab/router_detailed.py').write_bytes(base64.b64decode('IiIiUHJvYmUtZmlyc3QgcmFuayByb3V0aW5nLCBTRFBBIGV4YWN0IGZhbGxiYWNrLCBhbmQgZXhwbGljaXQgY2FjaGVkIGFzc2lnbm1lbnRzLgpUaGUgb2xkIHJvdXRlciBpcyByZXRhaW5lZCB1bmNoYW5nZWQgZm9yIGNvbXBhcmlzb25zLiBDYWNoZWQgcm91dGluZyBuZXZlciBtZWFucwphY2N1cmFjeSBpcyBndWFyYW50ZWVkIG9uIGNoYW5nZWQgaW5wdXRzLiBTZWxlY3Rpb24gYW5kIGV4ZWN1dGlvbiBhcmUgc2VwYXJhYmxlLgoiIiIKaW1wb3J0IHRvcmNoCmZyb20gLmFkYXB0aXZlIGltcG9ydCBvbWVnYSwgZmVhdHVyZXMsIGV4YWN0X3Jvd3MsIHJmX2F0dGVudGlvbgoKCmRlZiBzYW1wbGVkX3JmKHEsIGssIHYsIHcsIGlkeCwgY2F1c2FsPUZhbHNlLCBjaHVuaz0yNTYpOgogICAgIiIiRXZhbHVhdGUgb25seSByZXF1ZXN0ZWQgcm93czsgY2F1c2FsIHVzZXMgcHJlZml4IHN1ZmZpY2llbnQgc3RhdGlzdGljcy4iIiIKICAgIGtmID0gZmVhdHVyZXMoaywgdywgVHJ1ZSkKICAgIHFmID0gZmVhdHVyZXMocVsuLi4sIGlkeCwgOl0sIHcsIEZhbHNlKQogICAgaWYgbm90IGNhdXNhbDoKICAgICAgICBrdiA9IGtmLnRyYW5zcG9zZSgtMSwgLTIpIEAgdgogICAgICAgIHogPSBrZi5zdW0oLTIpLnVuc3F1ZWV6ZSgtMSkKICAgICAgICByZXR1cm4gKHFmIEAga3YpIC8gKHFmIEAgeikKICAgIHN0ID0gcS5uZXdfemVyb3MoKnEuc2hhcGVbOi0yXSwgdy5zaGFwZVswXSwgdi5zaGFwZVstMV0pCiAgICB6cyA9IHEubmV3X3plcm9zKCpxLnNoYXBlWzotMl0sIHcuc2hhcGVbMF0sIDEpCiAgICBvdXQgPSB0b3JjaC5lbXB0eV9saWtlKHFbLi4uLCBpZHgsIDpdKQogICAgZm9yIHMgaW4gcmFuZ2UoMCwgay5zaGFwZVstMl0sIGNodW5rKToKICAgICAgICBzdG9wID0gbWluKHMrY2h1bmssIGsuc2hhcGVbLTJdKQogICAgICAgIGtjLCB2YyA9IGtmWy4uLiwgczpzdG9wLCA6XSwgdlsuLi4sIHM6c3RvcCwgOl0KICAgICAgICBtYXNrID0gKGlkeCA+PSBzKSAmIChpZHggPCBzdG9wKQogICAgICAgIGlmIG1hc2suYW55KCk6CiAgICAgICAgICAgIGEgPSBxZlsuLi4sIG1hc2ssIDpdIEAga2MudHJhbnNwb3NlKC0xLC0yKQogICAgICAgICAgICBhID0gYS5tYXNrZWRfZmlsbCh0b3JjaC5hcmFuZ2UocyxzdG9wLGRldmljZT1xLmRldmljZSlbTm9uZSw6XSA+IGlkeFttYXNrLE5vbmVdLCAwKQogICAgICAgICAgICBvdXRbLi4uLG1hc2ssOl0gPSAocWZbLi4uLG1hc2ssOl0gQCBzdCArIGEgQCB2YykgLyAocWZbLi4uLG1hc2ssOl0gQCB6cyArIGEuc3VtKC0xLGtlZXBkaW09VHJ1ZSkpCiAgICAgICAgc3QgKz0ga2MudHJhbnNwb3NlKC0xLC0yKSBAIHZjCiAgICAgICAgenMgKz0ga2Muc3VtKC0yKS51bnNxdWVlemUoLTEpCiAgICByZXR1cm4gb3V0CgoKZGVmIGNob29zZShxLGssdix0b2w9LjEsbl9wcm9iZT0xNixjYXVzYWw9RmFsc2Usc2VlZD0wLHJhbmtzPSg2NCwyNTYsMTAyNCkpOgogICAgQixILE4sRD1xLnNoYXBlCiAgICBnPXRvcmNoLkdlbmVyYXRvcigpLm1hbnVhbF9zZWVkKHNlZWQpCiAgICBpZHg9dG9yY2gucmFuZHBlcm0oTixnZW5lcmF0b3I9ZylbOm1pbihuX3Byb2JlLE4pXS5zb3J0KCkudmFsdWVzLnRvKHEuZGV2aWNlKQogICAgcSxrLHY9KHguZmxvYXQoKS5yZXNoYXBlKEIqSCwxLHguc2hhcGVbLTJdLEQpIGZvciB4IGluIChxLGssdikpCiAgICBleGFjdD1leGFjdF9yb3dzKHFbLi4uLGlkeCw6XSxrLHYsaWR4IGlmIGNhdXNhbCBlbHNlIE5vbmUpCiAgICBhc3NpZ25tZW50PXRvcmNoLnplcm9zKEIqSCxkdHlwZT10b3JjaC5sb25nLGRldmljZT1xLmRldmljZSkKICAgIGVycm9ycz10b3JjaC5mdWxsKChCKkgsbGVuKHJhbmtzKSksZmxvYXQoJ25hbicpLGRldmljZT1xLmRldmljZSkKICAgIGZvciBqLG0gaW4gZW51bWVyYXRlKHJhbmtzKToKICAgICAgICBwZW5kaW5nPXRvcmNoLndoZXJlKGFzc2lnbm1lbnQ9PTApWzBdCiAgICAgICAgaWYgbm90IGxlbihwZW5kaW5nKTpicmVhawogICAgICAgIGdyb3VwPW1heCgxLDIqKjI2Ly8oay5zaGFwZVstMl0qbSkpCiAgICAgICAgZm9yIHN0IGluIHJhbmdlKDAsbGVuKHBlbmRpbmcpLGdyb3VwKToKICAgICAgICAgICAgc2VsPXBlbmRpbmdbc3Q6c3QrZ3JvdXBdCiAgICAgICAgICAgIGNhbmRpZGF0ZT1zYW1wbGVkX3JmKHFbc2VsXSxrW3NlbF0sdltzZWxdLG9tZWdhKG0sRCxxLmRldmljZSxzZWVkKSxpZHgsY2F1c2FsKQogICAgICAgICAgICBlcnI9KChjYW5kaWRhdGUtZXhhY3Rbc2VsXSkubm9ybShkaW09LTEpL2V4YWN0W3NlbF0ubm9ybShkaW09LTEpLmNsYW1wX21pbigxZS0zMCkpLm1lYW4oKC0xLC0yKSkKICAgICAgICAgICAgZXJyb3JzW3NlbCxqXT1lcnIKICAgICAgICAgICAgYXNzaWdubWVudFtzZWxbZXJyPD10b2xdXT1tCiAgICByZXR1cm4gYXNzaWdubWVudC5yZXNoYXBlKEIsSCksZXJyb3JzLnJlc2hhcGUoQixILGxlbihyYW5rcykpLGlkeAoKCmRlZiBleGVjdXRlKHEsayx2LGFzc2lnbm1lbnQsY2F1c2FsPUZhbHNlLHNlZWQ9MCxjaHVuaz0yNTYpOgogICAgQixILE4sRD1xLnNoYXBlCiAgICBxcixrcix2cj0oeC5yZXNoYXBlKEIqSCwxLHguc2hhcGVbLTJdLEQpIGZvciB4IGluIChxLGssdikpCiAgICBvdXQ9dG9yY2guZW1wdHlfbGlrZShxcikKICAgIGE9YXNzaWdubWVudC5yZXNoYXBlKC0xKQogICAgZm9yIG0gaW4gKDAsNjQsMjU2LDEwMjQpOgogICAgICAgIHNlbD10b3JjaC53aGVyZShhPT1tKVswXQogICAgICAgIGlmIG5vdCBsZW4oc2VsKTogY29udGludWUKICAgICAgICBpZiBtPT0wOgogICAgICAgICAgICBvdXRbc2VsXT10b3JjaC5ubi5mdW5jdGlvbmFsLnNjYWxlZF9kb3RfcHJvZHVjdF9hdHRlbnRpb24ocXJbc2VsXSxrcltzZWxdLHZyW3NlbF0saXNfY2F1c2FsPWNhdXNhbCkKICAgICAgICBlbHNlOgogICAgICAgICAgICBncm91cD1tYXgoMSwyKioyNi8vKE4qbSkpCiAgICAgICAgICAgIGZvciBzIGluIHJhbmdlKDAsbGVuKHNlbCksZ3JvdXApOgogICAgICAgICAgICAgICAgaXg9c2VsW3M6cytncm91cF0KICAgICAgICAgICAgICAgIHosXz1yZl9hdHRlbnRpb24ocXJbaXhdLmZsb2F0KCksa3JbaXhdLmZsb2F0KCksdnJbaXhdLmZsb2F0KCksb21lZ2EobSxELHEuZGV2aWNlLHNlZWQpLGNhdXNhbCxjaHVuaykKICAgICAgICAgICAgICAgIG91dFtpeF09ei50byhxLmR0eXBlKQogICAgcmV0dXJuIG91dC5yZXNoYXBlKEIsSCxOLEQpCgoKZGVmIHJvdXRlKHEsayx2LHRvbD0uMSxuX3Byb2JlPTE2LGNhdXNhbD1GYWxzZSxzZWVkPTApOgogICAgYSxfLF89Y2hvb3NlKHEsayx2LHRvbCxuX3Byb2JlLGNhdXNhbCxzZWVkKQogICAgcmV0dXJuIGV4ZWN1dGUocSxrLHYsYSxjYXVzYWwsc2VlZCksYQo='))
pathlib.Path('run_router_detailed.py').write_bytes(base64.b64decode('...'))
pathlib.Path('tests/test_router_detailed.py').write_bytes(base64.b64decode('aW1wb3J0IHRvcmNoCmZyb20gbGFiLmFkYXB0aXZlIGltcG9ydCByZl9hdHRlbnRpb24sb21lZ2EKZnJvbSBsYWIucm91dGVyX2RldGFpbGVkIGltcG9ydCBzYW1wbGVkX3JmLGV4ZWN1dGUsY2hvb3NlCgpkZWYgdGVzdF9zYW1wbGVkX21hdGNoZXNfZnVsbCgpOgogICAgZm9yIGNhdXNhbCBpbiAoRmFsc2UsVHJ1ZSk6CiAgICAgICAgdG9yY2gubWFudWFsX3NlZWQoNCkKICAgICAgICBxLGssdj0odG9yY2gucmFuZG4oMiwxLDQzLDgpKi4yIGZvciBfIGluIHJhbmdlKDMpKQogICAgICAgIGl4PXRvcmNoLnRlbnNvcihbMCw0LDE1LDE2LDE3LDQyXSk7dz1vbWVnYSg2NCw4LCdjcHUnKQogICAgICAgIGZ1bGwsXz1yZl9hdHRlbnRpb24ocSxrLHYsdyxjYXVzYWw9Y2F1c2FsLGNodW5rPTE2KQogICAgICAgIHBhcnQ9c2FtcGxlZF9yZihxLGssdix3LGl4LGNhdXNhbD1jYXVzYWwsY2h1bms9MTYpCiAgICAgICAgdG9yY2gudGVzdGluZy5hc3NlcnRfY2xvc2UocGFydCxmdWxsWy4uLixpeCw6XSxydG9sPTFlLTQsYXRvbD0xZS02KQoKZGVmIHRlc3RfZXhhY3RfZmFsbGJhY2soKToKICAgIHEsayx2PSh0b3JjaC5yYW5kbigxLDIsMTksOCkgZm9yIF8gaW4gcmFuZ2UoMykpCiAgICBmb3IgYyBpbiAoRmFsc2UsVHJ1ZSk6CiAgICAgICAgYWN0dWFsPWV4ZWN1dGUocSxrLHYsdG9yY2guemVyb3MoMSwyLGR0eXBlPXRvcmNoLmxvbmcpLGMpCiAgICAgICAgZXhwZWN0ZWQ9dG9yY2gubm4uZnVuY3Rpb25hbC5zY2FsZWRfZG90X3Byb2R1Y3RfYXR0ZW50aW9uKHEsayx2LGlzX2NhdXNhbD1jKQogICAgICAgIHRvcmNoLnRlc3RpbmcuYXNzZXJ0X2Nsb3NlKGFjdHVhbCxleHBlY3RlZCkKCmRlZiB0ZXN0X3plcm9fdG9sZXJhbmNlX3NlbGVjdHNfZXhhY3QoKToKICAgIHEsayx2PSh0b3JjaC5yYW5kbigxLDIsMTksOCkgZm9yIF8gaW4gcmFuZ2UoMykpCiAgICBhLGVycm9ycyxfPWNob29zZShxLGssdix0b2w9MCkKICAgIGFzc2VydCAoYT09MCkuYWxsKCkgYW5kIGVycm9ycy5zaGFwZT09KDEsMiwzKQoKZGVmIHRlc3RfcXdlbl9rdl9leHBhbnNpb25fY29udHJhY3QoKToKICAgIGZyb20gbGFiLnF3ZW5fcGF0Y2ggaW1wb3J0IF9leHBhbmRfa3YKICAgIHE9dG9yY2gucmFuZG4oMSw0LDExLDgpO2s9dG9yY2gucmFuZG4oMSwyLDExLDgpO3Y9dG9yY2gucmFuZG4oMSwyLDExLDgpCiAgICBrLHY9X2V4cGFuZF9rdihxLGssdikKICAgIGFzc2VydCBrLnNoYXBlPT12LnNoYXBlPT1xLnNoYXBlCg=='))
pathlib.Path('ROUTER_DETAILED_PLAN.md').write_bytes(base64.b64decode('UXVlc3Rpb246IERvZXMgYSBkZXRhaWxlZCBtZWFzdXJlZCBldmFsdWF0aW9uIGNoYW5nZSB0aGUgYWRhcHRpdmUtcmFuayByb3V0ZXIgdmVyZGljdD8KRml4ZWQgYmVmb3JlIG5ldyBHUFUgcmVzdWx0czogc2NhbGVzIC4xLC4yLC4yNSwuMywuNCwuNSwuNzUsMTsgaGV0ZXJvZ2VuZW91cyBzY2FsZXM7IGxvdy1yYW5rLCBjbHVzdGVyZWQsIGxvY2FsK2dsb2JhbCwgcmFyZSBzaGFycCBxdWVyaWVzLiBCb3RoIGNhdXNhbCBtb2Rlcy4gSW5kZXBlbmRlbnQgaGVsZC1vdXQgcXVlcnkgZXJyb3JzIGFuZCBmdWxsIGVycm9ycy4gU2VlZHMgMTIzNCwyMzQ1LDM0NTYuIFRvbGVyYW5jZXMgLjAxLC4wMjUsLjA1LC4wNzUsLjEsLjE1LC4yLC4zLC40OyBwcm9iZXMgNCw4LDE2LDMyLDY0LDEyOC4gTiA1MTIsMTAyNCwyMDQ4LDQwOTYsODE5MiwxNjM4NCwzMjc2OCw2NTUzNjsgbG9uZ2VzdCBncmlkIG9uIHNlbGVjdGVkIHByZXNwZWNpZmllZCByZWdpbWVzLiBPbmUtc2hvdCBvcmlnaW5hbCB2cyBwcm9iZS1maXJzdCBoeWJyaWQ7IGZpeGVkIHJhbmtzOyBTRFBBOyBjYWNoZWQgc2FtZS1pbnB1dCBhbmQgbmV4dC1pbnB1dCBkcmlmdC4gUmVhbCBwb3N0LVJvUEUgUXdlbiBhY3RpdmF0aW9uczsgbWF0Y2hlZCAxNiBXaWtpVGV4dCB3aW5kb3dzIGJhc2VsaW5lL29uZS1zaG90L2NhY2hlZCBoeWJyaWQsIHRvbGVyYW5jZXMgLjEsLjIsLjQuIE5vIENQVSBiZW5jaG1hcmssIG5vIGludmVudGVkIHJlc3VsdHMuCg=='))
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"],check=True)
subprocess.run(["python", "-m", "pytest", "-q", "tests"],check=True)
subprocess.run(["python", "run_router_detailed.py", "--parts", "qwen", "--out", "results/router_detailed_qwen"],check=True)
print("ROUTER_DETAILED_COMPLETE")
